# Discussion Log & Matrix Tactic Scale Generator

This notebook generates seed data for Round 1 games (G026-G250):
- `discussion_log`: Player dialogues reflecting their tactics
- `matrix_tactic_scale`: Mapping of player utterances to 4×4 behavior matrix
- `prior_summary_gold`: Empty for Round 1 (no prior history)

**Approach:**
1. Load dataset and filter Round 1 games needing generation
2. Build tactics knowledge base from research framework
3. Design LLM prompt templates with constraints
4. Generate seeds via OpenAI API with validation
5. Save and review results

In [1]:
import pandas as pd
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple
import time
from datetime import datetime

## Step 1: Load Dataset and Filter Round 1 Games

In [2]:
# Load the full dataset
df = pd.read_csv('Deception-Dataset.csv')

print(f"Total rows: {len(df)}")
print(f"Unique games: {df['game_id'].nunique()}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Total rows: 1000
Unique games: 250
Columns: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale', 'reasoning_gold', 'Overall_with_formula', 'Overall']

First few rows:


,game_id,round_id,role_id,llm_alignment,player_roles,public_history,prior_summary_gold,discussion_log,matrix_tactic_scale,reasoning_gold,Overall_with_formula,Overall
0,G001,1,P1,Good,"{""P1"":""Good"",\n ""P2"":""Good"",\n ""P3"":""Evil"",\...","Round: 1\nLeader: P4\nTeam: P1, P4\nVotes: P1:...","""""","Discussion after Quest 1:\n\nP3: ""I think that...","{\n ""P3"": {""row"":""Selective / Framing"",""col"":...","{\r\n ""abduction"":[\r\n {""player"":""P3"",""go...","{\n ""game_id"": ""G001"",\n ""round_id"": 1,\n ""...","{\n ""game_id"": ""G001"",\n ""round_id"": 1,\n ""..."
1,G002,1,P1,Good,"{""P1"":""Good"",\n ""P2"":""Good"",\n ""P3"":""Good"",\...","Round: 1\r\nLeader: P4\r\nTeam: P1, P4\r\nVote...","""""","Discussion after Quest 1:\n\nP3: ""P1 and P4 we...","{\n ""P3"": {""row"":""Transparent"",""col"":""Coopera...","{\r\n ""abduction"":[\r\n {""player"":""P3"",""go...","{\n ""game_id"": ""G002"",\n ""round_id"": 1,\n ""...","{\n ""game_id"": ""G002"",\n ""round_id"": 1,\n ""..."
2,G003,1,P1,Good,"{""P1"":""Good"",\n ""P2"":""Good"",\n ""P3"":""Good"",\...","Round: 1\r\nLeader: P4\r\nTeam: P1, P4\r\nVote...","""""","Discussion after Quest 1:\n\nP4: ""Maybe it's P...","{\n ""P4"": {""row"":""Counterfactual"",""col"":""Defe...","{\r\n ""abduction"":[\r\n {""player"":""P4"",""go...","{\n ""game_id"": ""G003"",\n ""round_id"": 1,\n ""...","{\n ""game_id"": ""G003"",\n ""round_id"": 1,\n ""..."
3,G004,1,P1,Good,"{""P1"":""Good"",\n ""P2"":""Good"",\n ""P3"":""Evil"",\...","Round: 1\r\nLeader: P4\r\nTeam: P1, P4\r\nVote...","""""","Discussion after Quest 1:\n\nP4: ""It wasn't me...","\n{\n ""P4"": {""row"":""Counterfactual"",""col"":""De...","{\r\n ""abduction"":[\r\n {""player"":""P3"",""go...","{\n ""game_id"": ""G004"",\n ""round_id"": 1,\n ""...","{\n ""game_id"": ""G004"",\n ""round_id"": 1,\n ""..."
4,G005,1,P1,Good,"{""P1"":""Good"",\n ""P2"":""Good"",\n ""P3"":""Evil"",\...","Round: 1\r\nLeader: P4\r\nTeam: P1, P4\r\nVote...","""""","Discussion after Quest 1:\n\nP2: ""Because the ...","{\n ""P2"": {""row"":""Selective / Framing"",""col"":...","{\r\n ""abduction"":[\r\n {""player"":""P2"",""go...","{\n ""game_id"": ""G005"",\n ""round_id"": 1,\n ""...","{\r\n ""game_id"": ""G005"",\r\n ""round_id"": 1,\..."


In [3]:
# Filter Round 1 games only
round1_df = df[df['round_id'] == 1].copy()

print(f"Total Round 1 rows: {len(round1_df)}")
print(f"Unique Round 1 games: {round1_df['game_id'].nunique()}")

# Check discussion_log status
# Note: In CSV, empty strings might be stored as empty quotes or actual empty cells
round1_df['has_discussion'] = round1_df['discussion_log'].notna() & (round1_df['discussion_log'].str.strip() != '')

games_with_discussion = round1_df[round1_df['has_discussion']]['game_id'].unique()
games_without_discussion = round1_df[~round1_df['has_discussion']]['game_id'].unique()

print(f"\nGames with discussion_log: {len(games_with_discussion)}")
print(f"Games without discussion_log: {len(games_without_discussion)}")
print(f"\nFirst 10 games without discussion: {sorted(games_without_discussion)[:10]}")

Total Round 1 rows: 250
Unique Round 1 games: 250

Games with discussion_log: 25
Games without discussion_log: 225

First 10 games without discussion: ['G026', 'G027', 'G028', 'G029', 'G030', 'G031', 'G032', 'G033', 'G034', 'G035']


In [5]:
# Filter Round 1 rows that need generation (no discussion_log)
# Each row represents one game's Round 1 with a specific protagonist (role_id)
# The protagonist does NOT speak in the discussion - only the other 4 players speak
games_to_generate = round1_df[~round1_df['has_discussion']].copy()

print(f"\nGames to generate: {len(games_to_generate)}")
print(f"Game IDs range: {games_to_generate['game_id'].min()} to {games_to_generate['game_id'].max()}")

# Display context columns we'll use for generation
print(f"\nContext columns available:")
context_cols = ['game_id', 'round_id', 'role_id', 'player_roles', 'public_history']
games_to_generate[context_cols].head()


Games to generate: 225
Game IDs range: G026 to G250

Context columns available:


,game_id,round_id,role_id,player_roles,public_history
25,G026,1,P1,"{""P1"":""Good"",\n ""P2"":""Good"",\n ""P3"":""Evil"",\...","Round: 1\r\nLeader: P3\r\nTeam: P1, P3\r\nVote..."
26,G027,1,P1,"{""P1"":""Good"",\n ""P2"":""Good"",\n ""P3"":""Good"",\...","Round: 1\r\nLeader: P5\r\nTeam: P1, P5\r\nVote..."
27,G028,1,P1,"{""P1"":""Good"",\n ""P2"":""Evil"",\n ""P3"":""Evil"",\...","Round: 1\r\nLeader: P1\r\nTeam: P1, P2\r\nVote..."
28,G029,1,P1,"{""P1"":""Good"",\n ""P2"":""Good"",\n ""P3"":""Good"",\...","Round: 1\r\nLeader: P4\r\nTeam: P1, P4\r\nVote..."
29,G030,1,P1,"{""P1"":""Good"",\n ""P2"":""Good"",\n ""P3"":""Good"",\...","Round: 1\r\nLeader: P2\r\nTeam: P1, P2\r\nVote..."


In [6]:
# Parse a sample to understand structure
sample_game = games_to_generate.iloc[0]

print(f"Sample Game: {sample_game['game_id']}")
print(f"\nPlayer Roles:")
player_roles = json.loads(sample_game['player_roles'])
for player, role in player_roles.items():
    print(f"  {player}: {role}")

print(f"\nPublic History:")
print(sample_game['public_history'])

# Identify Good and Evil players
good_players = [p for p, r in player_roles.items() if r == 'Good']
evil_players = [p for p, r in player_roles.items() if r == 'Evil']

print(f"\nGood players: {good_players}")
print(f"Evil players: {evil_players}")
print(f"\nProtagonist (role_id): {sample_game['role_id']} - should not speak in discussion")

Sample Game: G026

Player Roles:
  P1: Good
  P2: Good
  P3: Evil
  P4: Evil
  P5: Good

Public History:
Round: 1
Leader: P3
Team: P1, P3
Votes: P1:Y P2:Y P3:Y P4:Y P5:Y
Quest 1 Outcome: PASS

Good players: ['P1', 'P2', 'P5']
Evil players: ['P3', 'P4']

Protagonist (role_id): P1 - should not speak in discussion


## Step 2: Build Tactics Knowledge Base

Extract the 4×4 behavior matrix from the research framework:
- **Rows (Information Strategy)**: Transparent, Selective/Framing, Careless-to-truth, Counterfactual
- **Columns (Goal Orientation)**: Cooperative, Defensive, Opportunistic, Adversarial
- **Scales**: GRS (Goodness Rationality Spectrum) for Good players, Mach-IV for Evil players
- **Levels**: High, Moderate, Low

In [7]:
# Define the 4x4 behavior matrix structure
# Based on Table in deception.tex

BEHAVIOR_MATRIX = {
    ('Transparent', 'Cooperative'): {
        'tactics': ['Evidence sharing', 'Rational justification', 'Perspective-taking'],
        'alignment': 'Good',
        'color': 'green'
    },
    ('Transparent', 'Defensive'): {
        'tactics': ['Defensive disclosure', 'Self-justification'],
        'alignment': 'Both',
        'color': 'blue'
    },
    ('Transparent', 'Opportunistic'): {
        'tactics': ['Weaponized candor', 'Overstated contribution'],
        'alignment': 'Evil',
        'color': 'red'
    },
    ('Transparent', 'Adversarial'): {
        'tactics': ['Truthful misdirection', 'Misleading emphasis'],
        'alignment': 'Evil',
        'color': 'red'
    },
    ('Selective / Framing', 'Cooperative'): {
        'tactics': ['Bayesian hedging', 'Strategic uncertainty', 'Pragmatic silence'],
        'alignment': 'Good',
        'color': 'green'
    },
    ('Selective / Framing', 'Defensive'): {
        'tactics': ['Omission to save face', 'Half-truths'],
        'alignment': 'Both',
        'color': 'blue'
    },
    ('Selective / Framing', 'Opportunistic'): {
        'tactics': ['Cherry-picking', 'False consensus', 'Misframing'],
        'alignment': 'Evil',
        'color': 'red'
    },
    ('Selective / Framing', 'Adversarial'): {
        'tactics': ['Withholding info (WIFOM)', 'Strategic omission'],
        'alignment': 'Evil',
        'color': 'red'
    },
    ('Careless-to-truth', 'Cooperative'): {
        'tactics': ['Noisy sincerity', 'Over-explaining'],
        'alignment': 'Good',
        'color': 'green'
    },
    ('Careless-to-truth', 'Defensive'): {
        'tactics': ['Face-saving blather', 'Vague filler'],
        'alignment': 'Both',
        'color': 'blue'
    },
    ('Careless-to-truth', 'Opportunistic'): {
        'tactics': ['Bullshitting', 'Technobabble'],
        'alignment': 'Evil',
        'color': 'red'
    },
    ('Careless-to-truth', 'Adversarial'): {
        'tactics': ['Obfuscatory nonsense', 'Confusion tactics'],
        'alignment': 'Evil',
        'color': 'red'
    },
    ('Counterfactual', 'Cooperative'): {
        'tactics': ['Honest mistake', 'Sincere false belief'],
        'alignment': 'Good',
        'color': 'green'
    },
    ('Counterfactual', 'Defensive'): {
        'tactics': ['Deflection', 'Self-deception', 'Rationalized error'],
        'alignment': 'Both',
        'color': 'blue'
    },
    ('Counterfactual', 'Opportunistic'): {
        'tactics': ['Reverse accusations', 'Strategic fabrication'],
        'alignment': 'Evil',
        'color': 'red'
    },
    ('Counterfactual', 'Adversarial'): {
        'tactics': ['Hard lying', 'Gaslighting', 'Bold falsehoods'],
        'alignment': 'Evil',
        'color': 'red'
    }
}

print(f"Total matrix cells: {len(BEHAVIOR_MATRIX)}")
print(f"Total tactics: {sum(len(v['tactics']) for v in BEHAVIOR_MATRIX.values())}")

Total matrix cells: 16
Total tactics: 37


In [8]:
# Define tactic definitions from deception.tex
TACTIC_DEFINITIONS = {
    # Transparent row
    'Evidence sharing': 'providing concrete facts to support a claim',
    'Rational justification': 'explaining actions through logical reasons',
    'Perspective-taking': "acknowledging another's viewpoint to build trust",
    'Defensive disclosure': 'revealing only what is necessary to protect oneself',
    'Self-justification': "excusing one's own actions to avoid blame",
    'Weaponized candor': 'using blunt truth strategically to gain advantage',
    'Overstated contribution': "exaggerating one's role or effort",
    'Truthful misdirection': 'giving accurate details that nonetheless distract',
    'Misleading emphasis': 'highlighting true but minor points to distort perception',
    
    # Selective/Framing row
    'Bayesian hedging': 'presenting probabilities or uncertainty to remain flexible',
    'Strategic uncertainty': 'deliberately leaving outcomes vague',
    'Pragmatic silence': 'omitting information instead of lying outright',
    'Omission to save face': 'withholding details to avoid embarrassment',
    'Half-truths': 'mixing truth with key omissions',
    'Cherry-picking': 'selectively presenting only favorable evidence',
    'False consensus': 'overstating agreement to pressure others',
    'Misframing': 'presenting information in a skewed context',
    'Withholding info (WIFOM)': 'creating doubt by hiding critical details',
    'Strategic omission': 'leaving out facts to influence interpretation',
    
    # Careless-to-truth row
    'Noisy sincerity': 'being honest in a confusing or overloaded way',
    'Over-explaining': 'adding excessive detail to appear genuine',
    'Face-saving blather': 'using empty talk to avoid confrontation',
    'Vague filler': 'relying on imprecise or generic phrases',
    'Bullshitting': 'speaking without concern for truth or falsity',
    'Technobabble': 'using jargon to appear credible without substance',
    'Obfuscatory nonsense': 'intentionally confusing speech to mislead',
    'Confusion tactics': 'derailing clarity to disrupt understanding',
    
    # Counterfactual row
    'Honest mistake': 'presenting something false in good faith',
    'Sincere false belief': 'confidently holding a mistaken view',
    'Deflection': 'shifting blame or suspicion away from oneself by redirecting attention to another player or event, often using partial truths or emotional appeals',
    'Self-deception': 'convincing oneself of a falsehood',
    'Rationalized error': 'justifying a mistake with faulty logic',
    'Reverse accusations': 'shifting blame back onto the accuser',
    'Strategic fabrication': 'inventing details to achieve a goal',
    'Hard lying': 'deliberately asserting the opposite of the truth',
    'Gaslighting': 'making others doubt their memory or perception',
    'Bold falsehoods': 'telling obvious lies with confidence'
}

print(f"Total tactic definitions: {len(TACTIC_DEFINITIONS)}")

# Verify all tactics in matrix have definitions
all_tactics = set()
for cell_data in BEHAVIOR_MATRIX.values():
    all_tactics.update(cell_data['tactics'])

missing_definitions = all_tactics - set(TACTIC_DEFINITIONS.keys())
if missing_definitions:
    print(f"\n⚠️ Tactics missing definitions: {missing_definitions}")
else:
    print("\n✓ All tactics have definitions")

Total tactic definitions: 37

✓ All tactics have definitions


In [9]:
# Define scale assignments
SCALE_DEFINITIONS = {
    'GRS': {
        'name': 'Goodness Rationality Spectrum',
        'for': 'Good players',
        'levels': {
            'High': 'Highly rational, evidence-based, transparent communication',
            'Moderate': 'Generally rational with some uncertainty or hedging',
            'Low': 'Less structured reasoning, may be confused or overwhelmed'
        }
    },
    'Mach-IV': {
        'name': 'Machiavellianism Scale (Mach-IV)',
        'for': 'Evil players',
        'levels': {
            'High': 'Highly skilled manipulator, subtle deception, strategic planning',
            'Moderate': 'Competent deceiver, uses standard tactics, some slips',
            'Low': 'Clumsy deceiver, obvious tells, defensive, may contradict self'
        }
    }
}

print("Scale Definitions:")
print("\n=== GRS (Good Players) ===")
for level, desc in SCALE_DEFINITIONS['GRS']['levels'].items():
    print(f"  {level}: {desc}")

print("\n=== Mach-IV (Evil Players) ===")
for level, desc in SCALE_DEFINITIONS['Mach-IV']['levels'].items():
    print(f"  {level}: {desc}")

Scale Definitions:

=== GRS (Good Players) ===
  High: Highly rational, evidence-based, transparent communication
  Moderate: Generally rational with some uncertainty or hedging
  Low: Less structured reasoning, may be confused or overwhelmed

=== Mach-IV (Evil Players) ===
  High: Highly skilled manipulator, subtle deception, strategic planning
  Moderate: Competent deceiver, uses standard tactics, some slips
  Low: Clumsy deceiver, obvious tells, defensive, may contradict self


In [10]:
# Helper functions to filter tactics by player role
def get_valid_tactics_for_role(role: str) -> Dict[Tuple[str, str], List[str]]:
    """
    Get valid tactics for a player role.
    Good players: green cells only
    Evil players: red or blue cells
    """
    valid_tactics = {}
    
    for (row, col), cell_data in BEHAVIOR_MATRIX.items():
        if role == 'Good' and cell_data['color'] == 'green':
            valid_tactics[(row, col)] = cell_data['tactics']
        elif role == 'Evil' and cell_data['color'] in ['red', 'blue']:
            valid_tactics[(row, col)] = cell_data['tactics']
    
    return valid_tactics

def get_scale_for_role(role: str) -> str:
    """Get the appropriate scale for a player role."""
    return 'GRS' if role == 'Good' else 'Mach-IV'

# Test
print("Valid tactic cells for Good players:")
good_tactics = get_valid_tactics_for_role('Good')
print(f"  {len(good_tactics)} cells available")
print(f"  Total tactics: {sum(len(t) for t in good_tactics.values())}")

print("\nValid tactic cells for Evil players:")
evil_tactics = get_valid_tactics_for_role('Evil')
print(f"  {len(evil_tactics)} cells available")
print(f"  Total tactics: {sum(len(t) for t in evil_tactics.values())}")

Valid tactic cells for Good players:
  4 cells available
  Total tactics: 10

Valid tactic cells for Evil players:
  12 cells available
  Total tactics: 27


In [11]:
# Display a formatted view of the matrix
import pandas as pd

# Create a readable matrix view
rows = ['Transparent', 'Selective / Framing', 'Careless-to-truth', 'Counterfactual']
cols = ['Cooperative', 'Defensive', 'Opportunistic', 'Adversarial']

matrix_view = []
for row in rows:
    row_data = {'Row': row}
    for col in cols:
        cell_data = BEHAVIOR_MATRIX[(row, col)]
        tactics_str = ', '.join(cell_data['tactics'])
        row_data[col] = f"{cell_data['color']}: {tactics_str}"
    matrix_view.append(row_data)

matrix_df = pd.DataFrame(matrix_view)
print("4×4 Behavior Matrix:")
matrix_df

4×4 Behavior Matrix:


,Row,Cooperative,Defensive,Opportunistic,Adversarial
0,Transparent,"green: Evidence sharing, Rational justificatio...","blue: Defensive disclosure, Self-justification","red: Weaponized candor, Overstated contribution","red: Truthful misdirection, Misleading emphasis"
1,Selective / Framing,"green: Bayesian hedging, Strategic uncertainty...","blue: Omission to save face, Half-truths","red: Cherry-picking, False consensus, Misframing","red: Withholding info (WIFOM), Strategic omission"
2,Careless-to-truth,"green: Noisy sincerity, Over-explaining","blue: Face-saving blather, Vague filler","red: Bullshitting, Technobabble","red: Obfuscatory nonsense, Confusion tactics"
3,Counterfactual,"green: Honest mistake, Sincere false belief","blue: Deflection, Self-deception, Rationalized...","red: Reverse accusations, Strategic fabrication","red: Hard lying, Gaslighting, Bold falsehoods"


In [12]:
# Save knowledge base to JSON for reference
knowledge_base = {
    'behavior_matrix': {
        f"{row}|{col}": {
            'tactics': cell_data['tactics'],
            'alignment': cell_data['alignment'],
            'color': cell_data['color']
        }
        for (row, col), cell_data in BEHAVIOR_MATRIX.items()
    },
    'tactic_definitions': TACTIC_DEFINITIONS,
    'scale_definitions': SCALE_DEFINITIONS
}

with open('tactics_knowledge_base.json', 'w') as f:
    json.dump(knowledge_base, f, indent=2)

print("✓ Tactics knowledge base saved to: tactics_knowledge_base.json")

✓ Tactics knowledge base saved to: tactics_knowledge_base.json


## Summary

**Step 1 Complete:** 
- Loaded dataset and identified games needing generation
- Extracted context columns: `game_id`, `player_roles`, `public_history`
- Identified protagonist (role_id) who should NOT speak

**Step 2 Complete:**
- Built 4×4 behavior matrix with 24 tactics
- Defined all tactic descriptions
- Created scale definitions (GRS for Good, Mach-IV for Evil)
- Implemented helper functions to filter tactics by role
- Saved knowledge base to JSON

**Next Steps:**
- Step 3: Design LLM prompt template with suggested tactics + scales
- Step 4: Implement API caller with validation
- Step 5: Generate seeds with configurable loop

## Step 3: Pre-compute Tactic and Scale Assignments

Strategy: Ensure balanced coverage across all 24 tactics and skill levels (High/Moderate/Low)

In [13]:
import random
from collections import defaultdict

def assign_tactics_to_games(games_df):
    """
    Pre-assign tactics and scales to games for balanced coverage.
    
    Returns dict: {game_id: {player: {tactic, row, col, scale, level}}}
    """
    assignments = {}
    
    # Get all tactics by role
    good_tactics_flat = []
    evil_tactics_flat = []
    
    for (row, col), cell_data in BEHAVIOR_MATRIX.items():
        for tactic in cell_data['tactics']:
            tactic_info = {
                'tactic': tactic,
                'row': row,
                'col': col,
                'alignment': cell_data['alignment']
            }
            if cell_data['color'] == 'green':
                good_tactics_flat.append(tactic_info)
            elif cell_data['color'] in ['red', 'blue']:
                evil_tactics_flat.append(tactic_info)
    
    print(f"Good tactics pool: {len(good_tactics_flat)}")
    print(f"Evil tactics pool: {len(evil_tactics_flat)}")
    
    # Create cyclic iterators for balanced distribution
    random.shuffle(good_tactics_flat)
    random.shuffle(evil_tactics_flat)
    
    good_tactic_cycle = 0
    evil_tactic_cycle = 0
    
    levels = ['High', 'Moderate', 'Low']
    level_cycle = 0
    
    # Assign to each game
    for idx, row_data in games_df.iterrows():
        game_id = row_data['game_id']
        player_roles = json.loads(row_data['player_roles'])
        protagonist = row_data['role_id']
        
        game_assignments = {}
        
        # Assign tactics to each speaking player (exclude protagonist)
        for player in ['P1', 'P2', 'P3', 'P4', 'P5']:
            if player == protagonist:
                continue  # Protagonist doesn't speak
            
            role = player_roles[player]
            
            if role == 'Good':
                tactic_info = good_tactics_flat[good_tactic_cycle % len(good_tactics_flat)]
                good_tactic_cycle += 1
                scale = 'GRS'
            else:  # Evil
                tactic_info = evil_tactics_flat[evil_tactic_cycle % len(evil_tactics_flat)]
                evil_tactic_cycle += 1
                scale = 'Mach-IV'
            
            # Assign level cyclically
            level = levels[level_cycle % len(levels)]
            level_cycle += 1
            
            game_assignments[player] = {
                'tactic': tactic_info['tactic'],
                'row': tactic_info['row'],
                'col': tactic_info['col'],
                'scale': scale,
                'level': level
            }
        
        assignments[game_id] = game_assignments
    
    return assignments

# Pre-compute assignments
print("Pre-computing tactic and scale assignments...")
tactic_assignments = assign_tactics_to_games(games_to_generate)

print(f"\n✓ Assigned tactics to {len(tactic_assignments)} games")

# Show a sample assignment
sample_game_id = list(tactic_assignments.keys())[0]
print(f"\nSample assignment for {sample_game_id}:")
for player, assignment in tactic_assignments[sample_game_id].items():
    print(f"  {player}: {assignment['tactic']} ({assignment['row']}, {assignment['col']}) - {assignment['scale']} {assignment['level']}")

Pre-computing tactic and scale assignments...
Good tactics pool: 10
Evil tactics pool: 27

✓ Assigned tactics to 225 games

Sample assignment for G026:
  P2: Honest mistake (Counterfactual, Cooperative) - GRS High
  P3: False consensus (Selective / Framing, Opportunistic) - Mach-IV Moderate
  P4: Misleading emphasis (Transparent, Adversarial) - Mach-IV Low
  P5: Noisy sincerity (Careless-to-truth, Cooperative) - GRS High


In [14]:
# Analyze tactic coverage
tactic_usage = defaultdict(int)
scale_level_usage = defaultdict(int)

for game_id, players in tactic_assignments.items():
    for player, assignment in players.items():
        tactic_usage[assignment['tactic']] += 1
        scale_level_usage[f"{assignment['scale']}_{assignment['level']}"] += 1

print("Tactic Coverage (target: ~10 per tactic for 225 games with 4 speakers each = 900 total):")
print(f"Total assignments: {sum(tactic_usage.values())}")
print(f"\nTop 10 most used tactics:")
for tactic, count in sorted(tactic_usage.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {tactic}: {count}")

print(f"\nBottom 10 least used tactics:")
for tactic, count in sorted(tactic_usage.items(), key=lambda x: x[1])[:10]:
    print(f"  {tactic}: {count}")

print(f"\nScale-Level Distribution:")
for scale_level, count in sorted(scale_level_usage.items()):
    print(f"  {scale_level}: {count}")

Tactic Coverage (target: ~10 per tactic for 225 games with 4 speakers each = 900 total):
Total assignments: 900

Top 10 most used tactics:
  Honest mistake: 45
  Noisy sincerity: 45
  Rational justification: 45
  Bayesian hedging: 45
  Strategic uncertainty: 45
  Sincere false belief: 45
  Perspective-taking: 45
  Over-explaining: 45
  Pragmatic silence: 45
  Evidence sharing: 45

Bottom 10 least used tactics:
  Reverse accusations: 16
  Half-truths: 16
  Bullshitting: 16
  Confusion tactics: 16
  Self-deception: 16
  Rationalized error: 16
  Strategic omission: 16
  Hard lying: 16
  Vague filler: 16
  False consensus: 17

Scale-Level Distribution:
  GRS_High: 149
  GRS_Low: 148
  GRS_Moderate: 153
  Mach-IV_High: 151
  Mach-IV_Low: 152
  Mach-IV_Moderate: 147


## Step 4: Design LLM Prompt Template

In [27]:
def build_prompt(game_row, suggested_tactics):
    """
    Build LLM prompt with game context and suggested tactics.
    
    Args:
        game_row: DataFrame row with game data
        suggested_tactics: Dict of {player: {tactic, row, col, scale, level}}
    
    Returns:
        str: Complete prompt for LLM
    """
    game_id = game_row['game_id']
    player_roles = json.loads(game_row['player_roles'])
    public_history = game_row['public_history']
    protagonist = game_row['role_id']
    
    # Identify good and evil players
    good_players = [p for p, r in player_roles.items() if r == 'Good']
    evil_players = [p for p, r in player_roles.items() if r == 'Evil']
    
    # Format suggested tactics
    suggestions_text = []
    for player in ['P1', 'P2', 'P3', 'P4', 'P5']:
        if player == protagonist:
            continue
        assignment = suggested_tactics[player]
        suggestions_text.append(
            f"  {player} ({player_roles[player]}): Use tactic '{assignment['tactic']}' "
            f"({assignment['row']} × {assignment['col']}) with {assignment['scale']} scale at {assignment['level']} level"
        )
    
    prompt = f"""You are an expert in social deduction games and deception theory. Generate a realistic discussion log for Round 1 of an Avalon game based on the provided context.

# GAME CONTEXT

**Game ID:** {game_id}
**Round:** 1
**Protagonist (does NOT speak):** {protagonist}

**Player Roles:**
{chr(10).join(f'  - {p}: {r}' for p, r in player_roles.items())}

**Good Players:** {', '.join(good_players)}
**Evil Players:** {', '.join(evil_players)}

**Public History:**
{public_history}

# AVALON RULES

- Good players: Try to deduce evil players and pass quests
- Evil players: Sabotage quests while avoiding detection
- Round 1: First quest has completed, players discuss what happened
- The protagonist ({protagonist}) is observing but does NOT participate in this discussion

# 4×4 BEHAVIOR MATRIX

The matrix has 4 rows (Information Strategy) × 4 columns (Goal Orientation):

**Rows (Information Strategy):**
- Transparent: openly sharing clear, evidence-based information
- Selective/Framing: shaping communication through omission or emphasis
- Careless-to-truth: speaking without concern for factual accuracy
- Counterfactual: asserting or believing information contrary to reality

**Columns (Goal Orientation):**
- Cooperative: acting in alignment with group goals and mutual benefit
- Defensive: protecting one's own image, reputation, or position
- Opportunistic: exploiting situations for personal gain at the expense of others
- Adversarial: seeking to undermine, harm, or sabotage others

**Tactic Definitions:**
{chr(10).join(f'- {tactic}: {defn}' for tactic, defn in TACTIC_DEFINITIONS.items())}

# SCALE DEFINITIONS

**GRS (Goodness Rationality Spectrum) - For Good Players:**
- High: Highly rational, evidence-based, transparent communication
- Moderate: Generally rational with some uncertainty or hedging
- Low: Less structured reasoning, may be confused or overwhelmed

**Mach-IV (Machiavellianism Scale) - For Evil Players:**
- High: Highly skilled manipulator, subtle deception, strategic planning
- Moderate: Competent deceiver, uses standard tactics, some slips
- Low: Clumsy deceiver, obvious tells, defensive, may contradict self

# SUGGESTED TACTICS FOR THIS GAME

Use these suggested tactics for each speaker (you may adjust if it makes the dialogue more natural):

{chr(10).join(suggestions_text)}

# TASK

Generate a JSON response matching the EXACT format from the dataset examples.

1. **discussion_log**: A string with this EXACT format:
   ```
   Discussion after Quest 1:

   P#: "dialogue text"

   P#: "dialogue text"

   P#: "dialogue text"

   P#: "dialogue text"
   ```
   
   CRITICAL FORMAT RULES:
   - Start with "Discussion after Quest 1:" followed by blank line
   - Each speaker line: P# (no space before colon): "dialogue in double quotes"
   - Blank line between each speaker
   - NO extra blank line after the last speaker
   - Exactly 4 players speak (all except {protagonist})
   - Each player speaks once
   - Dialogue reflects their suggested tactic and skill level
   - Good players use cooperative/transparent tactics appropriate to their GRS level
   - Evil players use deceptive tactics appropriate to their Mach-IV level
   - Reference the quest outcome from public history
   - If the mission passed, then in general the Good players are confident but cautious, Evil players are defensive
   - If the mission failed, then in general the Good players are suspicious, Evil players are misleading or deflecting
   - Keep each statement 1-2 sentences, natural and conversational

2. **matrix_tactic_scale**: A JSON object with this EXACT structure:
   ```json
   {{
     "P#": {{"row":"Information Strategy","col":"Goal Orientation","tactic":"Tactic Name","scale":"GRS or Mach-IV","level":"High/Moderate/Low"}},
     ...
   }}
   ```
   
   CRITICAL FORMAT RULES:
   - Use exact row/col names from the matrix (e.g., "Selective / Framing", "Counterfactual")
   - Use exact tactic names (e.g., "False consensus", "Rational justification")
   - Scale must be "GRS" for Good players, "Mach-IV" for Evil players
   - Level must be "High", "Moderate", or "Low"
   - NO extra spaces in JSON (compact format)

# OUTPUT FORMAT

Return ONLY valid JSON with these two fields:

{{
  "discussion_log": "text of discussion with proper newlines",
  "matrix_tactic_scale": {{json object with player tactics}}
}}

The discussion_log should be plain text (not escaped), formatted like:
Discussion after Quest 1:

P#: "dialogue"

P#: "dialogue"

P#: "dialogue"

P#: "dialogue"

The matrix_tactic_scale should be a JSON object with compact formatting (no extra spaces):
{{
  "P#": {{"row":"...","col":"...","tactic":"...","scale":"...","level":"..."}},
  ...
}}

# EXAMPLES demonstrating the required format and style:

**Example 1:**
{{
  "discussion_log": "Discussion after Quest 1:\\n\\nP3: \\"I think that P1, P4, and me would make a great team\\"\\n\\nP2: \\"P1 and P4 went on the last mission and it failed. P3 must be evil\\"\\n\\nP4: \\"I know the first mission with me and P1 failed, but it was obviously P1 who failed it\\"\\n\\nP5: \\" It is not a good idea to form a team with both P1 and P4.\\"",
  "matrix_tactic_scale": {{
    "P3": {{"row":"Selective / Framing","col":"Opportunistic","tactic":"False consensus","scale":"Mach-IV","level":"Moderate"}},
    "P2": {{"row":"Transparent","col":"Cooperative","tactic":"Rational justification","scale":"GRS","level":"High"}},
    "P4": {{"row":"Counterfactual","col":"Defensive","tactic":"Deflection","scale":"Mach-IV","level":"High"}},
    "P5": {{"row":"Selective / Framing","col":"Cooperative","tactic":"Strategic uncertainty","scale":"GRS","level":"High"}}
  }}
}}

**Example 2:**
{{
  "discussion_log": "Discussion after Quest 1:\\n\\nP3: \\"Good job P1 and P4! The mission passed, so at least one of you is confirmed Good.\\"\\n\\nP2: \\"P1 and P4 seem trustworthy. We should build the next team around them.\\"\\n\\nP4: \\"Thanks. I think P1 is solid too. Let's keep working together.\\"\\n\\nP5: \\"I agree P1 looks good, but we shouldn't lock in teams too early. I want a chance to prove myself.\\"",
  "matrix_tactic_scale": {{
    "P3": {{"row":"Transparent","col":"Cooperative","tactic":"Evidence sharing","scale":"GRS","level":"High"}},
    "P2": {{"row":"Transparent","col":"Cooperative","tactic":"Rational justification","scale":"GRS","level":"High"}},
    "P4": {{"row":"Transparent","col":"Defensive","tactic":"Defensive disclosure","scale":"Mach-IV","level":"High"}},
    "P5": {{"row":"Selective / Framing","col":"Opportunistic","tactic":"False consensus","scale":"Mach-IV","level":"Moderate"}}
  }}
}}

**Example 3:**
{{
  "discussion_log": "Discussion after Quest 1:\\n\\nP2: \\"I think that P4, P5, and me would make a great team\\"\\n\\nP4: \\"I agree, as the last team failed it's better not to include any of them this time\\"\\n\\nP3: \\"I know the first mission with me failed, but it was obviously P1 who failed it, not me\\"\\n\\nP5: \\"It can be P3 too! And how can we believe?\\"",
  "matrix_tactic_scale": {{
    "P2": {{"row":"Transparent","col":"Cooperative","tactic":"Rational justification","scale":"GRS","level":"Low"}},
    "P4": {{"row":"Transparent","col":"Defensive","tactic":"Defensive disclosure","scale":"Mach-IV","level":"High"}},
    "P3": {{"row":"Counterfactual","col":"Adversarial","tactic":"Bold falsehoods","scale":"Mach-IV","level":"High"}},
    "P5": {{"row":"Transparent","col":"Cooperative","tactic":"Rational justification","scale":"GRS","level":"High"}}
  }}
}}

**Example 4:** 
{{
  "discussion_log": "Discussion after Quest 1:\\n\\nP2: \\"Let's pick P4, P5, and me. Safer than repeating P1 or P3.\\"\\n\\nP3: \\"I passed the mission. P1 must have failed it.\\"\\n\\nP5: \\"That can be lie. P3 may be trying to flip the story.\\"\\n\\nP4: \\"Both P1 and P3 are shady. We should drop both from future teams.\\"",
  "matrix_tactic_scale": {{
    "P2": {{"row":"Transparent","col":"Cooperative","tactic":"Evidence sharing","scale":"GRS","level":"High"}},
    "P3": {{"row":"Counterfactual","col":"Adversarial","tactic":"Hard lying","scale":"Mach-IV","level":"High"}},
    "P5": {{"row":"Selective / Framing","col":"Cooperative","tactic":"Strategic uncertainty","scale":"GRS","level":"High"}},
    "P4": {{"row":"Selective / Framing","col":"Opportunistic","tactic":"Cherry-picking","scale":"Mach-IV","level":"Moderate"}}
  }}
}}

**Example 5:**
{{
  "discussion_log": "Discussion after Quest 1:\\n\\nP2: \\"Let's continue with this team, as we passed and someone else for the next round.\\"\\n\\nP3: \\"It's great that we passed but we should be careful who to pick next, I vouch for myself.\\"\\n\\nP5: \\"Yay, the mission passed, but there may be hidden threats!\\"\\n\\nP4: \\"I am a good player, I should be on the next team.\\"",
  "matrix_tactic_scale": {{
    "P2": {{"row":"Transparent","col":"Cooperative","tactic":"Rational justification","scale":"GRS","level":"High"}},
    "P3": {{"row":"Transparent","col":"Defensive","tactic":"Self-justification","scale":"GRS","level":"Moderate"}},
    "P5": {{"row":"Careless-to-truth","col":"Defensive","tactic":"Vague filler","scale":"Mach-IV","level":"Moderate"}},
    "P4": {{"row":"Selective / Framing","col":"Opportunistic","tactic":"Misframing","scale":"Mach-IV","level":"Moderate"}}
  }}
}}

Generate your output following this exact format structure.
"""
    
    return prompt

# Test prompt generation
sample_game = games_to_generate.iloc[0]
sample_tactics = tactic_assignments[sample_game['game_id']]
sample_prompt = build_prompt(sample_game, sample_tactics)

print("Sample Prompt Preview (first 1000 chars):")
print(sample_prompt[:1000])
print("\n...")
print(f"\nTotal prompt length: {len(sample_prompt)} characters")

Sample Prompt Preview (first 1000 chars):
You are an expert in social deduction games and deception theory. Generate a realistic discussion log for Round 1 of an Avalon game based on the provided context.

# GAME CONTEXT

**Game ID:** G026
**Round:** 1
**Protagonist (does NOT speak):** P1

**Player Roles:**
  - P1: Good
  - P2: Good
  - P3: Evil
  - P4: Evil
  - P5: Good

**Good Players:** P1, P2, P5
**Evil Players:** P3, P4

**Public History:**
Round: 1
Leader: P3
Team: P1, P3
Votes: P1:Y P2:Y P3:Y P4:Y P5:Y
Quest 1 Outcome: PASS

# AVALON RULES

- Good players: Try to deduce evil players and pass quests
- Evil players: Sabotage quests while avoiding detection
- Round 1: First quest has completed, players discuss what happened
- The protagonist (P1) is observing but does NOT participate in this discussion

# 4×4 BEHAVIOR MATRIX

The matrix has 4 rows (Information Strategy) × 4 columns (Goal Orientation):

**Rows (Information Strategy):**
- Transparent: openly sharing clear, evidence-b

## Step 5: API Caller and Generation Loop

In [22]:
# Import existing LLM module
import sys
sys.path.append('.')
from llm import get_completion_with_backoff

print("✓ LLM module imported")
print("✓ get_completion_with_backoff supports all parameters via **kwargs")

✓ LLM module imported
✓ get_completion_with_backoff supports all parameters via **kwargs


In [23]:
def validate_generation(game_row, generated_data, suggested_tactics):
    """
    Validate generated discussion_log and matrix_tactic_scale match exact CSV format.
    
    Returns:
        tuple: (is_valid: bool, errors: list)
    """
    errors = []
    
    if not generated_data:
        return False, ["No data generated"]
    
    # Check required fields
    if 'discussion_log' not in generated_data:
        errors.append("Missing 'discussion_log' field")
    if 'matrix_tactic_scale' not in generated_data:
        errors.append("Missing 'matrix_tactic_scale' field")
    
    if errors:
        return False, errors
    
    discussion_log = generated_data['discussion_log']
    matrix_tactic_scale = generated_data['matrix_tactic_scale']
    
    protagonist = game_row['role_id']
    player_roles = json.loads(game_row['player_roles'])
    
    # Validate discussion_log format
    if not discussion_log.startswith("Discussion after Quest 1:"):
        errors.append("Discussion log must start with 'Discussion after Quest 1:'")
    
    # Check for protagonist not speaking
    if f"{protagonist}:" in discussion_log:
        errors.append(f"Protagonist {protagonist} appears to speak in discussion")
    
    # Count speakers in discussion log
    speaker_count = sum(1 for p in ['P1', 'P2', 'P3', 'P4', 'P5'] if f"{p}:" in discussion_log)
    if speaker_count != 4:
        errors.append(f"Expected 4 speakers in discussion, found {speaker_count}")
    
    # Validate matrix_tactic_scale structure
    speaking_players = [p for p in ['P1', 'P2', 'P3', 'P4', 'P5'] if p != protagonist]
    
    if len(matrix_tactic_scale) != 4:
        errors.append(f"Expected 4 speakers in matrix, got {len(matrix_tactic_scale)}")
    
    for player in speaking_players:
        if player not in matrix_tactic_scale:
            errors.append(f"Missing tactics for {player}")
            continue
        
        tactics = matrix_tactic_scale[player]
        required_fields = ['row', 'col', 'tactic', 'scale', 'level']
        for field in required_fields:
            if field not in tactics:
                errors.append(f"{player}: Missing '{field}' in tactics")
        
        # Validate scale matches role
        if 'scale' in tactics:
            expected_scale = 'GRS' if player_roles[player] == 'Good' else 'Mach-IV'
            if tactics['scale'] != expected_scale:
                errors.append(f"{player}: Wrong scale (expected {expected_scale}, got {tactics['scale']})")
        
        # Validate level
        if 'level' in tactics and tactics['level'] not in ['High', 'Moderate', 'Low']:
            errors.append(f"{player}: Invalid level '{tactics['level']}'")
        
        # Validate tactic exists in definitions
        if 'tactic' in tactics and tactics['tactic'] not in TACTIC_DEFINITIONS:
            errors.append(f"{player}: Unknown tactic '{tactics['tactic']}'")
    
    # Check protagonist doesn't appear in matrix
    if protagonist in matrix_tactic_scale:
        errors.append(f"Protagonist {protagonist} should not be in matrix_tactic_scale")
    
    return len(errors) == 0, errors

print("✓ Validation function defined")

✓ Validation function defined


## Generation Loop with Progress Saving

**Configure the number of games to generate below before running.**

In [32]:
# ============================================
# CONFIGURATION: Set number of games to generate
# ============================================
NUM_GAMES_TO_GENERATE = 225  # Start with 2 for testing, then increase to 225

print(f"Configuration: Will generate {NUM_GAMES_TO_GENERATE} games")
print(f"Total games available: {len(games_to_generate)}")

if NUM_GAMES_TO_GENERATE > len(games_to_generate):
    print(f"⚠️ Warning: Requested {NUM_GAMES_TO_GENERATE} but only {len(games_to_generate)} games available")
    NUM_GAMES_TO_GENERATE = len(games_to_generate)

Configuration: Will generate 225 games
Total games available: 225


In [30]:
# Main generation loop
output_file = 'generated_r1_seeds.csv'
results = []
failed_games = []

print(f"\n{'='*70}")
print(f"Starting generation of {NUM_GAMES_TO_GENERATE} games")
print(f"{'='*70}\n")

for idx in range(NUM_GAMES_TO_GENERATE):
    game_row = games_to_generate.iloc[idx]
    game_id = game_row['game_id']
    
    print(f"[{idx+1}/{NUM_GAMES_TO_GENERATE}] Generating {game_id}...", end=" ")
    
    # Get suggested tactics
    suggested_tactics = tactic_assignments[game_id]
    
    # Build prompt
    prompt = build_prompt(game_row, suggested_tactics)
    
    # Call LLM directly using get_completion_with_backoff from llm.py
    try:
        response = get_completion_with_backoff(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are an expert in social deduction games and deception theory. You generate realistic game discussions with tactical communication patterns."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.5,
            max_tokens=1024
        )
        
        content = response.choices[0].message.content
        generated_data = json.loads(content)
        
    except json.JSONDecodeError as e:
        print(f"✗ JSON parse error: {e}")
        failed_games.append((game_id, [f"JSON parse error: {e}"]))
        continue
    except Exception as e:
        print(f"✗ API error: {e}")
        failed_games.append((game_id, [f"API error: {e}"]))
        continue
    
    # Validate
    is_valid, errors = validate_generation(game_row, generated_data, suggested_tactics)
    
    if is_valid:
        print("✓")
        
        # Prepare result row
        result = {
            'game_id': game_id,
            'round_id': game_row['round_id'],
            'role_id': game_row['role_id'],
            'llm_alignment': game_row['llm_alignment'],
            'player_roles': game_row['player_roles'],
            'public_history': game_row['public_history'],
            'prior_summary_gold': '',  # Empty for Round 1
            'discussion_log': generated_data['discussion_log'],
            'matrix_tactic_scale': json.dumps(generated_data['matrix_tactic_scale']),
        }
        results.append(result)
        
        # Save incrementally every 10 games
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(output_file, index=False)
            print(f"  💾 Progress saved ({len(results)} games)")
    
    else:
        print(f"✗ Validation failed")
        for error in errors:
            print(f"    - {error}")
        failed_games.append((game_id, errors))
    
    # Rate limiting
    time.sleep(1)

# Final save
if results:
    final_df = pd.DataFrame(results)
    final_df.to_csv(output_file, index=False)
    print(f"\n{'='*70}")
    print(f"✓ Generation complete!")
    print(f"  Successfully generated: {len(results)} games")
    print(f"  Failed: {len(failed_games)} games")
    print(f"  Output saved to: {output_file}")
    print(f"{'='*70}")
else:
    print("\n⚠️ No successful generations")

# Display failed games if any
if failed_games:
    print(f"\nFailed games:")
    for game_id, errors in failed_games:
        print(f"  {game_id}: {errors}")


Starting generation of 2 games

[1/2] Generating G026... ✓
[2/2] Generating G027... ✓

✓ Generation complete!
  Successfully generated: 2 games
  Failed: 0 games
  Output saved to: generated_r1_seeds.csv


In [31]:
# Review generated results
if results:
    results_df = pd.DataFrame(results)
    print(f"Generated {len(results_df)} rows")
    print(f"\nFirst generated game:")
    print(f"Game ID: {results_df.iloc[0]['game_id']}")
    print(f"\nDiscussion Log:")
    print(results_df.iloc[0]['discussion_log'])
    print(f"\nMatrix Tactic Scale:")
    print(json.dumps(json.loads(results_df.iloc[0]['matrix_tactic_scale']), indent=2))

Generated 2 rows

First generated game:
Game ID: G026

Discussion Log:
Discussion after Quest 1:

P2: "It's great that the mission passed, but I might have misjudged P3. Maybe they're not as suspicious as I thought."

P3: "See? We all voted yes, and it worked out. Let's keep this positive momentum going!"

P4: "Sure, the mission passed, but that doesn't mean we can trust everyone on it. We should be careful with our next picks."

P5: "I'm glad the mission passed, but let's not get too comfortable. We need to stay sharp and watch for any signs of sabotage."

Matrix Tactic Scale:
{
  "P2": {
    "row": "Counterfactual",
    "col": "Cooperative",
    "tactic": "Honest mistake",
    "scale": "GRS",
    "level": "High"
  },
  "P3": {
    "row": "Selective / Framing",
    "col": "Opportunistic",
    "tactic": "False consensus",
    "scale": "Mach-IV",
    "level": "Moderate"
  },
  "P4": {
    "row": "Transparent",
    "col": "Adversarial",
    "tactic": "Misleading emphasis",
    "scale": "

## Summary

**Steps 3-5 Complete:**

**Step 3: Tactic Assignment**
- Pre-computed balanced tactic + scale assignments for all 225 games
- Ensures coverage across all 24 tactics and skill levels
- Cyclic distribution for fairness

**Step 4: Prompt Design**
- Dynamic prompt template with game context
- Includes suggested tactics for each speaker
- Structured JSON output format
- Uses existing llm.py for API calls

**Step 5: Generation Loop**
- Configurable number of iterations (set `NUM_GAMES_TO_GENERATE`)
- One-at-a-time generation with validation
- Incremental CSV saving every 10 games
- Progress tracking and error reporting

**Usage:**
1. Set `NUM_GAMES_TO_GENERATE = 2` for testing
2. Review generated outputs
3. If satisfied, set `NUM_GAMES_TO_GENERATE = 225` for full batch
4. Results saved to `generated_r1_seeds.csv`